In [3]:
"""
xLSTM for BiH air-quality forecasting.

Wires into the real bih_shared.py (from shared_setup.ipynb), which exposes:
    load_split, load_groups, load_windows, load_coords, load_panel,
    target, context, mase_scale, score, aggregate

Key thing this respects that a naive port wouldn't: eval_windows.csv is
VALIDATION-ONLY (origins are drawn only from 2024, midnight, one per day).
There is no equivalent frozen file for training windows, so training
origins are built here with the *same* eligibility rule the notebook uses
for eval (min_real_target_hours, some real history in the preceding week),
just applied to the 2021-2023 train range instead. That keeps train/val
windowing philosophy identical without inventing a second convention.
"""

import os
import sys
import subprocess
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ---- pull the shared artifacts + the clean dataset, exactly as the other
# tracks do -------------------------------------------------------------
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
import gdown

SHARED_DIR = "shared"
gdown.download_folder(id="1ivElb2AifDNvIg47Ii6dV_XP9aO14WSE", output=SHARED_DIR, quiet=True)
sys.path.append(SHARED_DIR)
import bih_shared as bs

CLEAN_CSV = "bih_hourly_clean.csv"
if not os.path.exists(CLEAN_CSV):
    gdown.download(id="1QyfzghimgyRhQ735k42JetDEoK5kfGZJ", output=CLEAN_CSV, quiet=False)


# ---------------------------------------------------------------------------
# 1. Data loading — reuse the frozen split/partitions, build train origins
#    with the same rule the notebook used for eval, applied to 2021-2023
# ---------------------------------------------------------------------------

def build_train_origins(panel, split, station, pollutant, horizon):
    """Mirrors shared_setup.ipynb cell 14's eligibility rule (target-side
    only: >= min_real_target_hours real+unfilled hours in the target, some
    finite value somewhere in the preceding week), but over the train range
    instead of 2024, so training and eval share one windowing philosophy."""
    s = panel[(station, pollutant)]
    y = s.y.to_numpy()
    filled = s.filled.to_numpy()
    real = ~np.isnan(y) & ~filled

    stride = split["origin_stride_hours"]
    hour_of_day = split["origin_hour_of_day"]
    min_real_target = split["masking"]["min_real_target_hours"]

    train_start = pd.Timestamp(split["train"]["start"])
    train_end = pd.Timestamp(split["train"]["end"])
    candidates = s.index[(s.index >= train_start) & (s.index <= train_end)
                          & (s.index.hour == hour_of_day)]

    origins = []
    for ts in candidates:
        o = s.index.get_loc(ts)
        if o + horizon > len(y):
            continue
        if real[o:o + horizon].sum() < min_real_target:
            continue
        if not np.isfinite(y[max(0, o - 168):o]).any():
            continue
        origins.append(ts)
    return origins


class WindowDataset(Dataset):
    """One item = one forecast origin. Input is `lookback` hours of context
    (bs.context — filled values allowed there), target is `horizon` hours
    plus the mask of which of those hours are real+unfilled (bs.target)."""

    def __init__(self, panel, station, pollutant, origins, lookback, horizon,
                 fill_value=None):
        self.panel = panel
        self.station = station
        self.pollutant = pollutant
        self.origins = origins
        self.lookback = lookback
        self.horizon = horizon
        # impute remaining context NaNs with the series' own train mean
        # (context is meant to allow filled values, but real gaps > 3h can
        # still be NaN — the model can't consume those directly)
        s = panel[(station, pollutant)]
        self.fill_value = fill_value if fill_value is not None else float(np.nanmean(s.y.to_numpy()))

    def __len__(self):
        return len(self.origins)

    def __getitem__(self, idx):
        origin = self.origins[idx]
        ctx = bs.context(self.panel, self.station, self.pollutant, origin, self.lookback)

        # bs.context() returns fewer than `lookback` hours when the origin is
        # close to the start of the panel (not enough history yet, e.g. early
        # 2021). Left-pad with NaN so it's short-context, not wrong-context —
        # the observed-flag channel below marks the padding as unobserved,
        # same as it marks real gaps.
        if len(ctx) < self.lookback:
            pad = np.full(self.lookback - len(ctx), np.nan, dtype=ctx.dtype)
            ctx = np.concatenate([pad, ctx])

        observed = np.isfinite(ctx).astype(np.float32)
        ctx = np.nan_to_num(ctx, nan=self.fill_value).astype(np.float32)

        y, mask = bs.target(self.panel, self.station, self.pollutant, origin, self.horizon)
        y = np.nan_to_num(y, nan=0.0).astype(np.float32)

        x = np.stack([ctx, observed], axis=-1)  # (lookback, 2): value + observed-flag
        return (torch.from_numpy(x), torch.from_numpy(y),
                torch.from_numpy(mask.astype(np.float32)))


# ---------------------------------------------------------------------------
# 2. xLSTM blocks — simplified sLSTM (scalar memory, exponential gating)
#    and mLSTM (matrix memory) per Beck et al. 2024, stacked alternately
# ---------------------------------------------------------------------------

class sLSTMCell(nn.Module):
    """Scalar-memory LSTM cell with exponential input/forget gating and a
    stabilizer state (m_t), as in the xLSTM paper. Good for spike/plume-like
    dynamics (e.g. SO2)."""

    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.W = nn.Linear(input_size, 4 * hidden_size)
        self.R = nn.Linear(hidden_size, 4 * hidden_size, bias=False)

    def forward(self, x_t, state):
        h, c, n, m = state
        gates = self.W(x_t) + self.R(h)
        z, i_tilde, f_tilde, o_tilde = gates.chunk(4, dim=-1)

        z = torch.tanh(z)
        o = torch.sigmoid(o_tilde)

        # stabilized exponential gating
        m_new = torch.maximum(f_tilde + m, i_tilde)
        i = torch.exp(i_tilde - m_new)
        f = torch.exp(f_tilde + m - m_new)

        c_new = f * c + i * z
        n_new = f * n + i
        h_new = o * (c_new / n_new.clamp_min(1e-6))

        return h_new, (h_new, c_new, n_new, m_new)

    def init_state(self, batch_size, device):
        z = torch.zeros(batch_size, self.hidden_size, device=device)
        return (z, z, z, z)


class mLSTMCell(nn.Module):
    """Matrix-memory LSTM cell (parallelizable, larger storage capacity).
    Good default for the smoother diurnal patterns (PM10, NO2, O3)."""

    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.Wq = nn.Linear(input_size, hidden_size)
        self.Wk = nn.Linear(input_size, hidden_size)
        self.Wv = nn.Linear(input_size, hidden_size)
        self.Wi = nn.Linear(input_size, 1)
        self.Wf = nn.Linear(input_size, 1)
        self.Wo = nn.Linear(input_size, hidden_size)

    def forward(self, x_t, state):
        C, n, m = state  # C: (B,H,H) matrix memory, n: (B,H), m: (B,1) stabilizer

        q = self.Wq(x_t)
        k = self.Wk(x_t) / math.sqrt(self.hidden_size)
        v = self.Wv(x_t)
        i_tilde = self.Wi(x_t)
        f_tilde = self.Wf(x_t)
        o = torch.sigmoid(self.Wo(x_t))

        m_new = torch.maximum(f_tilde + m, i_tilde)
        i = torch.exp(i_tilde - m_new)
        f = torch.exp(f_tilde + m - m_new)

        C_new = f.unsqueeze(-1) * C + i.unsqueeze(-1) * torch.einsum("bh,bk->bhk", v, k)
        n_new = f * n + i * k

        h = o * torch.einsum("bhk,bk->bh", C_new, q) / \
            torch.einsum("bh,bh->b", n_new, q).clamp_min(1e-6).unsqueeze(-1)

        return h, (C_new, n_new, m_new)

    def init_state(self, batch_size, device):
        C = torch.zeros(batch_size, self.hidden_size, self.hidden_size, device=device)
        n = torch.zeros(batch_size, self.hidden_size, device=device)
        m = torch.zeros(batch_size, 1, device=device)
        return (C, n, m)


class xLSTMBlock(nn.Module):
    """One residual block wrapping either an sLSTM or mLSTM cell, run over
    the full sequence."""

    def __init__(self, input_size, hidden_size, kind="m"):
        super().__init__()
        self.kind = kind
        self.cell = mLSTMCell(input_size, hidden_size) if kind == "m" else sLSTMCell(input_size, hidden_size)
        self.norm = nn.LayerNorm(input_size)
        self.proj = nn.Linear(hidden_size, input_size) if hidden_size != input_size else nn.Identity()

    def forward(self, x):
        # x: (B, T, D)
        B, T, D = x.shape
        state = self.cell.init_state(B, x.device)
        x_norm = self.norm(x)
        outs = []
        for t in range(T):
            h, state = self.cell(x_norm[:, t], state)
            outs.append(h)
        h_seq = torch.stack(outs, dim=1)
        return x + self.proj(h_seq)  # residual


class xLSTMForecaster(nn.Module):
    def __init__(self, n_features, hidden_size=64, horizon=24, block_types=("m", "s", "m", "s")):
        super().__init__()
        self.in_proj = nn.Linear(n_features, hidden_size)
        self.blocks = nn.ModuleList([
            xLSTMBlock(hidden_size, hidden_size, kind=k) for k in block_types
        ])
        self.head = nn.Linear(hidden_size, horizon)

    def forward(self, x):
        # x: (B, lookback, n_features)
        h = self.in_proj(x)
        for block in self.blocks:
            h = block(h)
        return self.head(h[:, -1])  # (B, horizon)


# ---------------------------------------------------------------------------
# 3. Training + evaluation, scored the same way as baseline_metrics.csv
# ---------------------------------------------------------------------------

def masked_l1_loss(pred, y, mask):
    """MAE over real+unfilled target hours only — matches split.json's
    masking rule and bs.score's own masking."""
    err = torch.abs(pred - y) * mask
    return err.sum() / mask.sum().clamp_min(1)


def train_one_station(station, pollutant, lookback=168, horizon=24,
                       hidden_size=64, epochs=20, batch_size=64, lr=1e-3, device="cpu"):

    split = bs.load_split(SHARED_DIR)
    groups = bs.load_groups(SHARED_DIR)
    status = groups["by_station"].get(station, {}).get(pollutant)
    if status != "ok":
        raise ValueError(f"{station}-{pollutant} is '{status}', not eligible for training")

    panel = bs.load_panel(CLEAN_CSV, SHARED_DIR)
    windows = bs.load_windows(SHARED_DIR)  # 2024 only — this is the eval set

    train_origins = build_train_origins(panel, split, station, pollutant, horizon)
    val_origins = windows[(windows.station == station) & (windows.pollutant == pollutant)] \
        .origin.tolist()

    train_ds = WindowDataset(panel, station, pollutant, train_origins, lookback, horizon)
    val_ds = WindowDataset(panel, station, pollutant, val_origins, lookback, horizon,
                            fill_value=train_ds.fill_value)  # reuse train stats, no val leakage
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    model = xLSTMForecaster(n_features=2, hidden_size=hidden_size, horizon=horizon).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        total_loss, total_n = 0.0, 0
        for x, y, mask in train_dl:
            x, y, mask = x.to(device), y.to(device), mask.to(device)
            opt.zero_grad()
            pred = model(x)
            loss = masked_l1_loss(pred, y, mask)
            loss.backward()
            opt.step()
            total_loss += loss.item() * x.size(0)
            total_n += x.size(0)
        print(f"[{station}-{pollutant}] epoch {epoch+1}: train MAE={total_loss/total_n:.4f}")

    return model, val_ds, panel


def evaluate(model, val_ds, panel, station, pollutant, device="cpu"):
    """Scores with bs.score/bs.mase_scale directly, so numbers are computed
    identically to baseline_metrics.csv and comparable to it row-for-row."""
    model.eval()
    rows = []
    val_dl = DataLoader(val_ds, batch_size=64, shuffle=False)

    all_preds = []
    with torch.no_grad():
        for x, y, mask in val_dl:
            pred = model(x.to(device)).cpu().numpy()
            all_preds.append(pred)
    all_preds = np.concatenate(all_preds)

    for i, origin in enumerate(val_ds.origins):
        y, mask = bs.target(panel, station, pollutant, origin, val_ds.horizon)
        ctx = bs.context(panel, station, pollutant, origin, 512)  # same lookback as baselines
        scale = bs.mase_scale(ctx)
        rows.append({"model": "xlstm", "station": station, "pollutant": pollutant,
                     "origin": origin, **bs.score(all_preds[i], y, mask, scale)})

    result = pd.DataFrame(rows)
    per_pol = bs.aggregate(result.to_dict("records"))

    baseline_row = "persistence_last" if pollutant == "so2" else "persistence_t24"
    print(f"xLSTM vs the bar it needs to clear ({baseline_row}):")
    print(per_pol)
    print(f"(compare against baseline_metrics.csv, pollutant={pollutant}, "
          f"model={baseline_row})")
    return result, per_pol


if __name__ == "__main__":
    STATION, POLLUTANT, LOOKBACK, HORIZON = "Bjelave", "pm10", 168, 24
    model, val_ds, panel = train_one_station(STATION, POLLUTANT, lookback=LOOKBACK, horizon=HORIZON)
    result, summary = evaluate(model, val_ds, panel, STATION, POLLUTANT)

/content/shared/bih_shared.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "filled": g[f"{p}_filled"].fillna(False).to_numpy(bool)},
/content/shared/bih_shared.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "filled": g[f"{p}_filled"].fillna(False).to_numpy(bool)},
/content/shared/bih_shared.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcas

[Bjelave-pm10] epoch 1: train MAE=10760.2264
[Bjelave-pm10] epoch 2: train MAE=33.0388
[Bjelave-pm10] epoch 3: train MAE=32.8095
[Bjelave-pm10] epoch 4: train MAE=32.7088
[Bjelave-pm10] epoch 5: train MAE=32.6434
[Bjelave-pm10] epoch 6: train MAE=32.6033
[Bjelave-pm10] epoch 7: train MAE=32.5771
[Bjelave-pm10] epoch 8: train MAE=32.5432
[Bjelave-pm10] epoch 9: train MAE=32.4997
[Bjelave-pm10] epoch 10: train MAE=32.4507
[Bjelave-pm10] epoch 11: train MAE=32.4211
[Bjelave-pm10] epoch 12: train MAE=32.3841
[Bjelave-pm10] epoch 13: train MAE=32.3539
[Bjelave-pm10] epoch 14: train MAE=32.2926
[Bjelave-pm10] epoch 15: train MAE=32.2495
[Bjelave-pm10] epoch 16: train MAE=32.2033
[Bjelave-pm10] epoch 17: train MAE=32.1746
[Bjelave-pm10] epoch 18: train MAE=32.1153
[Bjelave-pm10] epoch 19: train MAE=32.0757
[Bjelave-pm10] epoch 20: train MAE=32.0266
xLSTM vs the bar it needs to clear (persistence_t24):
                       mae       rmse      mase
model pollutant                             

OPET

In [4]:
"""
xLSTM for BiH air-quality forecasting.

Wires into the real bih_shared.py (from shared_setup.ipynb), which exposes:
    load_split, load_groups, load_windows, load_coords, load_panel,
    target, context, mase_scale, score, aggregate

Key thing this respects that a naive port wouldn't: eval_windows.csv is
VALIDATION-ONLY (origins are drawn only from 2024, midnight, one per day).
There is no equivalent frozen file for training windows, so training
origins are built here with the *same* eligibility rule the notebook uses
for eval (min_real_target_hours, some real history in the preceding week),
just applied to the 2021-2023 train range instead. That keeps train/val
windowing philosophy identical without inventing a second convention.
"""

import os
import sys
import subprocess
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ---- pull the shared artifacts + the clean dataset, exactly as the other
# tracks do -------------------------------------------------------------
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
import gdown

SHARED_DIR = "shared"
gdown.download_folder(id="1ivElb2AifDNvIg47Ii6dV_XP9aO14WSE", output=SHARED_DIR, quiet=True)
sys.path.append(SHARED_DIR)
import bih_shared as bs

CLEAN_CSV = "bih_hourly_clean.csv"
if not os.path.exists(CLEAN_CSV):
    gdown.download(id="1QyfzghimgyRhQ735k42JetDEoK5kfGZJ", output=CLEAN_CSV, quiet=False)


# ---------------------------------------------------------------------------
# 1. Data loading — reuse the frozen split/partitions, build train origins
#    with the same rule the notebook used for eval, applied to 2021-2023
# ---------------------------------------------------------------------------

def build_train_origins(panel, split, station, pollutant, horizon):
    """Mirrors shared_setup.ipynb cell 14's eligibility rule (target-side
    only: >= min_real_target_hours real+unfilled hours in the target, some
    finite value somewhere in the preceding week), but over the train range
    instead of 2024, so training and eval share one windowing philosophy."""
    s = panel[(station, pollutant)]
    y = s.y.to_numpy()
    filled = s.filled.to_numpy()
    real = ~np.isnan(y) & ~filled

    stride = split["origin_stride_hours"]
    hour_of_day = split["origin_hour_of_day"]
    min_real_target = split["masking"]["min_real_target_hours"]

    train_start = pd.Timestamp(split["train"]["start"])
    train_end = pd.Timestamp(split["train"]["end"])
    candidates = s.index[(s.index >= train_start) & (s.index <= train_end)
                          & (s.index.hour == hour_of_day)]

    origins = []
    for ts in candidates:
        o = s.index.get_loc(ts)
        if o + horizon > len(y):
            continue
        if real[o:o + horizon].sum() < min_real_target:
            continue
        if not np.isfinite(y[max(0, o - 168):o]).any():
            continue
        origins.append(ts)
    return origins


class WindowDataset(Dataset):
    """One item = one forecast origin. Input is `lookback` hours of context
    (bs.context — filled values allowed there), target is `horizon` hours
    plus the mask of which of those hours are real+unfilled (bs.target).

    Values are standardized with train-set mu/sigma before hitting the
    model — xLSTM's exponential gating (exp(i_tilde - m_new)) is sensitive
    to input scale before the stabilizer state settles, and raw PM10/NO2/etc
    magnitudes (tens to hundreds) push it toward collapsing to the mean."""

    def __init__(self, panel, station, pollutant, origins, lookback, horizon, mu, sigma):
        self.panel = panel
        self.station = station
        self.pollutant = pollutant
        self.origins = origins
        self.lookback = lookback
        self.horizon = horizon
        self.mu = mu
        self.sigma = sigma if sigma > 1e-6 else 1.0

    def __len__(self):
        return len(self.origins)

    def __getitem__(self, idx):
        origin = self.origins[idx]
        ctx = bs.context(self.panel, self.station, self.pollutant, origin, self.lookback)

        # bs.context() returns fewer than `lookback` hours when the origin is
        # close to the start of the panel (not enough history yet, e.g. early
        # 2021). Left-pad with NaN so it's short-context, not wrong-context —
        # the observed-flag channel below marks the padding as unobserved,
        # same as it marks real gaps.
        if len(ctx) < self.lookback:
            pad = np.full(self.lookback - len(ctx), np.nan, dtype=ctx.dtype)
            ctx = np.concatenate([pad, ctx])

        observed = np.isfinite(ctx).astype(np.float32)
        ctx_norm = (ctx - self.mu) / self.sigma
        ctx_norm = np.nan_to_num(ctx_norm, nan=0.0).astype(np.float32)  # 0 == mean, in normalized space

        y, mask = bs.target(self.panel, self.station, self.pollutant, origin, self.horizon)
        y_norm = np.nan_to_num((y - self.mu) / self.sigma, nan=0.0).astype(np.float32)

        x = np.stack([ctx_norm, observed], axis=-1)  # (lookback, 2): normalized value + observed-flag
        return (torch.from_numpy(x), torch.from_numpy(y_norm),
                torch.from_numpy(mask.astype(np.float32)))


# ---------------------------------------------------------------------------
# 2. xLSTM blocks — simplified sLSTM (scalar memory, exponential gating)
#    and mLSTM (matrix memory) per Beck et al. 2024, stacked alternately
# ---------------------------------------------------------------------------

class sLSTMCell(nn.Module):
    """Scalar-memory LSTM cell with exponential input/forget gating and a
    stabilizer state (m_t), as in the xLSTM paper. Good for spike/plume-like
    dynamics (e.g. SO2)."""

    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.W = nn.Linear(input_size, 4 * hidden_size)
        self.R = nn.Linear(hidden_size, 4 * hidden_size, bias=False)

    def forward(self, x_t, state):
        h, c, n, m = state
        gates = self.W(x_t) + self.R(h)
        z, i_tilde, f_tilde, o_tilde = gates.chunk(4, dim=-1)

        z = torch.tanh(z)
        o = torch.sigmoid(o_tilde)

        # stabilized exponential gating
        m_new = torch.maximum(f_tilde + m, i_tilde)
        i = torch.exp(i_tilde - m_new)
        f = torch.exp(f_tilde + m - m_new)

        c_new = f * c + i * z
        n_new = f * n + i
        h_new = o * (c_new / n_new.clamp_min(1e-6))

        return h_new, (h_new, c_new, n_new, m_new)

    def init_state(self, batch_size, device):
        z = torch.zeros(batch_size, self.hidden_size, device=device)
        return (z, z, z, z)


class mLSTMCell(nn.Module):
    """Matrix-memory LSTM cell (parallelizable, larger storage capacity).
    Good default for the smoother diurnal patterns (PM10, NO2, O3)."""

    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.Wq = nn.Linear(input_size, hidden_size)
        self.Wk = nn.Linear(input_size, hidden_size)
        self.Wv = nn.Linear(input_size, hidden_size)
        self.Wi = nn.Linear(input_size, 1)
        self.Wf = nn.Linear(input_size, 1)
        self.Wo = nn.Linear(input_size, hidden_size)

    def forward(self, x_t, state):
        C, n, m = state  # C: (B,H,H) matrix memory, n: (B,H), m: (B,1) stabilizer

        q = self.Wq(x_t)
        k = self.Wk(x_t) / math.sqrt(self.hidden_size)
        v = self.Wv(x_t)
        i_tilde = self.Wi(x_t)
        f_tilde = self.Wf(x_t)
        o = torch.sigmoid(self.Wo(x_t))

        m_new = torch.maximum(f_tilde + m, i_tilde)
        i = torch.exp(i_tilde - m_new)
        f = torch.exp(f_tilde + m - m_new)

        C_new = f.unsqueeze(-1) * C + i.unsqueeze(-1) * torch.einsum("bh,bk->bhk", v, k)
        n_new = f * n + i * k

        h = o * torch.einsum("bhk,bk->bh", C_new, q) / \
            torch.einsum("bh,bh->b", n_new, q).clamp_min(1e-6).unsqueeze(-1)

        return h, (C_new, n_new, m_new)

    def init_state(self, batch_size, device):
        C = torch.zeros(batch_size, self.hidden_size, self.hidden_size, device=device)
        n = torch.zeros(batch_size, self.hidden_size, device=device)
        m = torch.zeros(batch_size, 1, device=device)
        return (C, n, m)


class xLSTMBlock(nn.Module):
    """One residual block wrapping either an sLSTM or mLSTM cell, run over
    the full sequence."""

    def __init__(self, input_size, hidden_size, kind="m"):
        super().__init__()
        self.kind = kind
        self.cell = mLSTMCell(input_size, hidden_size) if kind == "m" else sLSTMCell(input_size, hidden_size)
        self.norm = nn.LayerNorm(input_size)
        self.proj = nn.Linear(hidden_size, input_size) if hidden_size != input_size else nn.Identity()

    def forward(self, x):
        # x: (B, T, D)
        B, T, D = x.shape
        state = self.cell.init_state(B, x.device)
        x_norm = self.norm(x)
        outs = []
        for t in range(T):
            h, state = self.cell(x_norm[:, t], state)
            outs.append(h)
        h_seq = torch.stack(outs, dim=1)
        return x + self.proj(h_seq)  # residual


class xLSTMForecaster(nn.Module):
    def __init__(self, n_features, hidden_size=64, horizon=24, block_types=("m", "s", "m", "s")):
        super().__init__()
        self.in_proj = nn.Linear(n_features, hidden_size)
        self.blocks = nn.ModuleList([
            xLSTMBlock(hidden_size, hidden_size, kind=k) for k in block_types
        ])
        self.head = nn.Linear(hidden_size, horizon)

    def forward(self, x):
        # x: (B, lookback, n_features)
        h = self.in_proj(x)
        for block in self.blocks:
            h = block(h)
        return self.head(h[:, -1])  # (B, horizon)


# ---------------------------------------------------------------------------
# 3. Training + evaluation, scored the same way as baseline_metrics.csv
# ---------------------------------------------------------------------------

def masked_l1_loss(pred, y, mask):
    """MAE over real+unfilled target hours only — matches split.json's
    masking rule and bs.score's own masking."""
    err = torch.abs(pred - y) * mask
    return err.sum() / mask.sum().clamp_min(1)


def train_one_station(station, pollutant, lookback=168, horizon=24,
                       hidden_size=64, epochs=20, batch_size=64, lr=1e-3, device="cpu"):

    split = bs.load_split(SHARED_DIR)
    groups = bs.load_groups(SHARED_DIR)
    status = groups["by_station"].get(station, {}).get(pollutant)
    if status != "ok":
        raise ValueError(f"{station}-{pollutant} is '{status}', not eligible for training")

    panel = bs.load_panel(CLEAN_CSV, SHARED_DIR)
    windows = bs.load_windows(SHARED_DIR)  # 2024 only — this is the eval set

    # standardize using TRAIN period only (2021-2023) — using 2024 here would
    # leak validation-period statistics into training, same principle as the
    # PM2.5/PM10 ratio being fit on train only in shared_setup.ipynb cell 18
    s = panel[(station, pollutant)]
    train_start, train_end = pd.Timestamp(split["train"]["start"]), pd.Timestamp(split["train"]["end"])
    train_vals = s.y.to_numpy()[(s.index >= train_start) & (s.index <= train_end)]
    mu, sigma = float(np.nanmean(train_vals)), float(np.nanstd(train_vals))

    train_origins = build_train_origins(panel, split, station, pollutant, horizon)
    val_origins = windows[(windows.station == station) & (windows.pollutant == pollutant)] \
        .origin.tolist()

    train_ds = WindowDataset(panel, station, pollutant, train_origins, lookback, horizon, mu, sigma)
    val_ds = WindowDataset(panel, station, pollutant, val_origins, lookback, horizon, mu, sigma)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    model = xLSTMForecaster(n_features=2, hidden_size=hidden_size, horizon=horizon).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        total_loss, total_n = 0.0, 0
        for x, y, mask in train_dl:
            x, y, mask = x.to(device), y.to(device), mask.to(device)
            opt.zero_grad()
            pred = model(x)
            loss = masked_l1_loss(pred, y, mask)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # xLSTM's exponential
            opt.step()                                                       # gates need this
            total_loss += loss.item() * x.size(0)
            total_n += x.size(0)
        print(f"[{station}-{pollutant}] epoch {epoch+1}: train MAE (normalized)={total_loss/total_n:.4f}")

    return model, val_ds, panel, mu, sigma


def evaluate(model, val_ds, panel, station, pollutant, mu, sigma, device="cpu"):
    """Scores with bs.score/bs.mase_scale directly, so numbers are computed
    identically to baseline_metrics.csv and comparable to it row-for-row.
    Predictions are unnormalized back to native units first — mu/sigma are
    only a training-time convenience, baseline_metrics.csv and bs.score()
    both expect native-unit values (e.g. µg/m3)."""
    model.eval()
    rows = []
    val_dl = DataLoader(val_ds, batch_size=64, shuffle=False)

    all_preds = []
    with torch.no_grad():
        for x, y, mask in val_dl:
            pred_norm = model(x.to(device)).cpu().numpy()
            all_preds.append(pred_norm * sigma + mu)
    all_preds = np.concatenate(all_preds)

    for i, origin in enumerate(val_ds.origins):
        y, mask = bs.target(panel, station, pollutant, origin, val_ds.horizon)
        ctx = bs.context(panel, station, pollutant, origin, 512)  # same lookback as baselines
        scale = bs.mase_scale(ctx)
        rows.append({"model": "xlstm", "station": station, "pollutant": pollutant,
                     "origin": origin, **bs.score(all_preds[i], y, mask, scale)})

    result = pd.DataFrame(rows)
    per_pol = bs.aggregate(result.to_dict("records"))

    baseline_row = "persistence_last" if pollutant == "so2" else "persistence_t24"
    print(f"xLSTM vs the bar it needs to clear ({baseline_row}):")
    print(per_pol)
    print(f"(compare against baseline_metrics.csv, pollutant={pollutant}, "
          f"model={baseline_row})")
    return result, per_pol


if __name__ == "__main__":
    STATION, POLLUTANT, LOOKBACK, HORIZON = "Bjelave", "pm10", 168, 24
    model, val_ds, panel, mu, sigma = train_one_station(STATION, POLLUTANT, lookback=LOOKBACK, horizon=HORIZON)
    result, summary = evaluate(model, val_ds, panel, STATION, POLLUTANT, mu, sigma)

/content/shared/bih_shared.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "filled": g[f"{p}_filled"].fillna(False).to_numpy(bool)},
/content/shared/bih_shared.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "filled": g[f"{p}_filled"].fillna(False).to_numpy(bool)},
/content/shared/bih_shared.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcas

[Bjelave-pm10] epoch 1: train MAE (normalized)=32335.1726
[Bjelave-pm10] epoch 2: train MAE (normalized)=0.4541
[Bjelave-pm10] epoch 3: train MAE (normalized)=0.4287
[Bjelave-pm10] epoch 4: train MAE (normalized)=0.4205
[Bjelave-pm10] epoch 5: train MAE (normalized)=0.4148
[Bjelave-pm10] epoch 6: train MAE (normalized)=0.4124
[Bjelave-pm10] epoch 7: train MAE (normalized)=0.4086
[Bjelave-pm10] epoch 8: train MAE (normalized)=0.4083
[Bjelave-pm10] epoch 9: train MAE (normalized)=0.4066
[Bjelave-pm10] epoch 10: train MAE (normalized)=0.4067
[Bjelave-pm10] epoch 11: train MAE (normalized)=0.4052
[Bjelave-pm10] epoch 12: train MAE (normalized)=0.4044
[Bjelave-pm10] epoch 13: train MAE (normalized)=0.4031
[Bjelave-pm10] epoch 14: train MAE (normalized)=0.4042
[Bjelave-pm10] epoch 15: train MAE (normalized)=0.4030
[Bjelave-pm10] epoch 16: train MAE (normalized)=0.4001
[Bjelave-pm10] epoch 17: train MAE (normalized)=6.4803
[Bjelave-pm10] epoch 18: train MAE (normalized)=0.3997
[Bjelave-pm10] 

In [2]:
"""
xLSTM for BiH air-quality forecasting.

Wires into the real bih_shared.py (from shared_setup.ipynb), which exposes:
    load_split, load_groups, load_windows, load_coords, load_panel,
    target, context, mase_scale, score, aggregate

Key thing this respects that a naive port wouldn't: eval_windows.csv is
VALIDATION-ONLY (origins are drawn only from 2024, midnight, one per day).
There is no equivalent frozen file for training windows, so training
origins are built here with the *same* eligibility rule the notebook uses
for eval (min_real_target_hours, some real history in the preceding week),
just applied to the 2021-2023 train range instead. That keeps train/val
windowing philosophy identical without inventing a second convention.
"""

import os
import sys
import subprocess
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ---- pull the shared artifacts + the clean dataset, exactly as the other
# tracks do -------------------------------------------------------------
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
import gdown

SHARED_DIR = "shared"
gdown.download_folder(id="1ivElb2AifDNvIg47Ii6dV_XP9aO14WSE", output=SHARED_DIR, quiet=True)
sys.path.append(SHARED_DIR)
import bih_shared as bs

CLEAN_CSV = "bih_hourly_clean.csv"
if not os.path.exists(CLEAN_CSV):
    gdown.download(id="1QyfzghimgyRhQ735k42JetDEoK5kfGZJ", output=CLEAN_CSV, quiet=False)


# ---------------------------------------------------------------------------
# 1. Data loading — reuse the frozen split/partitions, build train origins
#    with the same rule the notebook used for eval, applied to 2021-2023
# ---------------------------------------------------------------------------

def build_train_origins(panel, split, station, pollutant, horizon):
    """Mirrors shared_setup.ipynb cell 14's eligibility rule (target-side
    only: >= min_real_target_hours real+unfilled hours in the target, some
    finite value somewhere in the preceding week), but over the train range
    instead of 2024, so training and eval share one windowing philosophy."""
    s = panel[(station, pollutant)]
    y = s.y.to_numpy()
    filled = s.filled.to_numpy()
    real = ~np.isnan(y) & ~filled

    stride = split["origin_stride_hours"]
    hour_of_day = split["origin_hour_of_day"]
    min_real_target = split["masking"]["min_real_target_hours"]

    train_start = pd.Timestamp(split["train"]["start"])
    train_end = pd.Timestamp(split["train"]["end"])
    candidates = s.index[(s.index >= train_start) & (s.index <= train_end)
                          & (s.index.hour == hour_of_day)]

    origins = []
    for ts in candidates:
        o = s.index.get_loc(ts)
        if o + horizon > len(y):
            continue
        if real[o:o + horizon].sum() < min_real_target:
            continue
        if not np.isfinite(y[max(0, o - 168):o]).any():
            continue
        origins.append(ts)
    return origins


class WindowDataset(Dataset):
    """One item = one forecast origin. Input is `lookback` hours of context
    (bs.context — filled values allowed there), target is `horizon` hours
    plus the mask of which of those hours are real+unfilled (bs.target).

    Values are standardized with train-set mu/sigma before hitting the
    model — xLSTM's exponential gating (exp(i_tilde - m_new)) is sensitive
    to input scale before the stabilizer state settles, and raw PM10/NO2/etc
    magnitudes (tens to hundreds) push it toward collapsing to the mean."""

    def __init__(self, panel, station, pollutant, origins, lookback, horizon, mu, sigma):
        self.panel = panel
        self.station = station
        self.pollutant = pollutant
        self.origins = origins
        self.lookback = lookback
        self.horizon = horizon
        self.mu = mu
        self.sigma = sigma if sigma > 1e-6 else 1.0

    def __len__(self):
        return len(self.origins)

    def __getitem__(self, idx):
        origin = self.origins[idx]
        ctx = bs.context(self.panel, self.station, self.pollutant, origin, self.lookback)

        # bs.context() returns fewer than `lookback` hours when the origin is
        # close to the start of the panel (not enough history yet, e.g. early
        # 2021). Left-pad with NaN so it's short-context, not wrong-context —
        # the observed-flag channel below marks the padding as unobserved,
        # same as it marks real gaps.
        if len(ctx) < self.lookback:
            pad = np.full(self.lookback - len(ctx), np.nan, dtype=ctx.dtype)
            ctx = np.concatenate([pad, ctx])

        observed = np.isfinite(ctx).astype(np.float32)
        ctx_norm = (ctx - self.mu) / self.sigma
        ctx_norm = np.nan_to_num(ctx_norm, nan=0.0).astype(np.float32)  # 0 == mean, in normalized space

        y, mask = bs.target(self.panel, self.station, self.pollutant, origin, self.horizon)
        y_norm = np.nan_to_num((y - self.mu) / self.sigma, nan=0.0).astype(np.float32)

        x = np.stack([ctx_norm, observed], axis=-1)  # (lookback, 2): normalized value + observed-flag
        return (torch.from_numpy(x), torch.from_numpy(y_norm),
                torch.from_numpy(mask.astype(np.float32)))


# ---------------------------------------------------------------------------
# 2. xLSTM blocks — simplified sLSTM (scalar memory, exponential gating)
#    and mLSTM (matrix memory) per Beck et al. 2024, stacked alternately
# ---------------------------------------------------------------------------

class sLSTMCell(nn.Module):
    """Scalar-memory LSTM cell with exponential input/forget gating and a
    stabilizer state (m_t), as in the xLSTM paper. Good for spike/plume-like
    dynamics (e.g. SO2)."""

    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.W = nn.Linear(input_size, 4 * hidden_size)
        self.R = nn.Linear(hidden_size, 4 * hidden_size, bias=False)

    def forward(self, x_t, state):
        h, c, n, m = state
        gates = self.W(x_t) + self.R(h)
        z, i_tilde, f_tilde, o_tilde = gates.chunk(4, dim=-1)

        z = torch.tanh(z)
        o = torch.sigmoid(o_tilde)

        # stabilized exponential gating
        m_new = torch.maximum(f_tilde + m, i_tilde)
        i = torch.exp(i_tilde - m_new)
        f = torch.exp(f_tilde + m - m_new)

        c_new = f * c + i * z
        n_new = f * n + i
        h_new = o * (c_new / n_new.clamp_min(1e-6))

        return h_new, (h_new, c_new, n_new, m_new)

    def init_state(self, batch_size, device):
        z = torch.zeros(batch_size, self.hidden_size, device=device)
        return (z, z, z, z)


class mLSTMCell(nn.Module):
    """Matrix-memory LSTM cell (parallelizable, larger storage capacity).
    Good default for the smoother diurnal patterns (PM10, NO2, O3)."""

    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.Wq = nn.Linear(input_size, hidden_size)
        self.Wk = nn.Linear(input_size, hidden_size)
        self.Wv = nn.Linear(input_size, hidden_size)
        self.Wi = nn.Linear(input_size, 1)
        self.Wf = nn.Linear(input_size, 1)
        self.Wo = nn.Linear(input_size, hidden_size)

    def forward(self, x_t, state):
        C, n, m = state  # C: (B,H,H) matrix memory, n: (B,H), m: (B,1) stabilizer

        q = self.Wq(x_t)
        k = self.Wk(x_t) / math.sqrt(self.hidden_size)
        v = self.Wv(x_t)
        i_tilde = self.Wi(x_t)
        f_tilde = self.Wf(x_t)
        o = torch.sigmoid(self.Wo(x_t))

        m_new = torch.maximum(f_tilde + m, i_tilde)
        i = torch.exp(i_tilde - m_new)
        f = torch.exp(f_tilde + m - m_new)

        C_new = f.unsqueeze(-1) * C + i.unsqueeze(-1) * torch.einsum("bh,bk->bhk", v, k)
        n_new = f * n + i * k

        h = o * torch.einsum("bhk,bk->bh", C_new, q) / \
            torch.einsum("bh,bh->b", n_new, q).clamp_min(1e-6).unsqueeze(-1)

        return h, (C_new, n_new, m_new)

    def init_state(self, batch_size, device):
        C = torch.zeros(batch_size, self.hidden_size, self.hidden_size, device=device)
        n = torch.zeros(batch_size, self.hidden_size, device=device)
        m = torch.zeros(batch_size, 1, device=device)
        return (C, n, m)


class xLSTMBlock(nn.Module):
    """One residual block wrapping either an sLSTM or mLSTM cell, run over
    the full sequence."""

    def __init__(self, input_size, hidden_size, kind="m"):
        super().__init__()
        self.kind = kind
        self.cell = mLSTMCell(input_size, hidden_size) if kind == "m" else sLSTMCell(input_size, hidden_size)
        self.norm = nn.LayerNorm(input_size)
        self.proj = nn.Linear(hidden_size, input_size) if hidden_size != input_size else nn.Identity()

    def forward(self, x):
        # x: (B, T, D)
        B, T, D = x.shape
        state = self.cell.init_state(B, x.device)
        x_norm = self.norm(x)
        outs = []
        for t in range(T):
            h, state = self.cell(x_norm[:, t], state)
            outs.append(h)
        h_seq = torch.stack(outs, dim=1)
        return x + self.proj(h_seq)  # residual


class xLSTMForecaster(nn.Module):
    def __init__(self, n_features, hidden_size=64, horizon=24, block_types=("m", "s", "m", "s")):
        super().__init__()
        self.in_proj = nn.Linear(n_features, hidden_size)
        self.blocks = nn.ModuleList([
            xLSTMBlock(hidden_size, hidden_size, kind=k) for k in block_types
        ])
        self.head = nn.Linear(hidden_size, horizon)

    def forward(self, x):
        # x: (B, lookback, n_features)
        h = self.in_proj(x)
        for block in self.blocks:
            h = block(h)
        return self.head(h[:, -1])  # (B, horizon)


# ---------------------------------------------------------------------------
# 3. Training + evaluation, scored the same way as baseline_metrics.csv
# ---------------------------------------------------------------------------

def masked_l1_loss(pred, y, mask):
    """MAE over real+unfilled target hours only — matches split.json's
    masking rule and bs.score's own masking."""
    err = torch.abs(pred - y) * mask
    return err.sum() / mask.sum().clamp_min(1)


def train_one_station(station, pollutant, lookback=168, horizon=24,
                       hidden_size=64, epochs=20, batch_size=64, lr=1e-3, device="cpu"):

    split = bs.load_split(SHARED_DIR)
    groups = bs.load_groups(SHARED_DIR)
    status = groups["by_station"].get(station, {}).get(pollutant)
    if status != "ok":
        raise ValueError(f"{station}-{pollutant} is '{status}', not eligible for training")

    panel = bs.load_panel(CLEAN_CSV, SHARED_DIR)
    windows = bs.load_windows(SHARED_DIR)  # 2024 only — this is the eval set

    # standardize using TRAIN period only (2021-2023) — using 2024 here would
    # leak validation-period statistics into training, same principle as the
    # PM2.5/PM10 ratio being fit on train only in shared_setup.ipynb cell 18
    s = panel[(station, pollutant)]
    train_start, train_end = pd.Timestamp(split["train"]["start"]), pd.Timestamp(split["train"]["end"])
    train_vals = s.y.to_numpy()[(s.index >= train_start) & (s.index <= train_end)]
    mu, sigma = float(np.nanmean(train_vals)), float(np.nanstd(train_vals))

    train_origins = build_train_origins(panel, split, station, pollutant, horizon)
    val_origins = windows[(windows.station == station) & (windows.pollutant == pollutant)] \
        .origin.tolist()

    train_ds = WindowDataset(panel, station, pollutant, train_origins, lookback, horizon, mu, sigma)
    val_ds = WindowDataset(panel, station, pollutant, val_origins, lookback, horizon, mu, sigma)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    model = xLSTMForecaster(n_features=2, hidden_size=hidden_size, horizon=horizon).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        total_loss, total_n = 0.0, 0
        for x, y, mask in train_dl:
            x, y, mask = x.to(device), y.to(device), mask.to(device)
            opt.zero_grad()
            pred = model(x)
            loss = masked_l1_loss(pred, y, mask)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # xLSTM's exponential
            opt.step()                                                       # gates need this
            total_loss += loss.item() * x.size(0)
            total_n += x.size(0)
        print(f"[{station}-{pollutant}] epoch {epoch+1}: train MAE (normalized)={total_loss/total_n:.4f}")

    return model, val_ds, panel, mu, sigma


def evaluate(model, val_ds, panel, station, pollutant, mu, sigma, device="cpu"):
    """Scores with bs.score/bs.mase_scale directly, so numbers are computed
    identically to baseline_metrics.csv and comparable to it row-for-row.
    Predictions are unnormalized back to native units first — mu/sigma are
    only a training-time convenience, baseline_metrics.csv and bs.score()
    both expect native-unit values (e.g. µg/m3)."""
    model.eval()
    rows = []
    val_dl = DataLoader(val_ds, batch_size=64, shuffle=False)

    all_preds = []
    with torch.no_grad():
        for x, y, mask in val_dl:
            pred_norm = model(x.to(device)).cpu().numpy()
            all_preds.append(pred_norm * sigma + mu)
    all_preds = np.concatenate(all_preds)

    for i, origin in enumerate(val_ds.origins):
        y, mask = bs.target(panel, station, pollutant, origin, val_ds.horizon)
        ctx = bs.context(panel, station, pollutant, origin, 512)  # same lookback as baselines
        scale = bs.mase_scale(ctx)
        rows.append({"model": "xlstm", "station": station, "pollutant": pollutant,
                     "origin": origin, **bs.score(all_preds[i], y, mask, scale)})

    result = pd.DataFrame(rows)
    per_pol = bs.aggregate(result.to_dict("records"))

    baseline_row = "persistence_last" if pollutant == "so2" else "persistence_t24"
    print(f"xLSTM vs the bar it needs to clear ({baseline_row}):")
    print(per_pol)
    print(f"(compare against baseline_metrics.csv, pollutant={pollutant}, "
          f"model={baseline_row})")
    return result, per_pol


if __name__ == "__main__":
    STATION, POLLUTANT, LOOKBACK, HORIZON = "Bjelave", "pm10", 168, 24
    model, val_ds, panel, mu, sigma = train_one_station(STATION, POLLUTANT, lookback=LOOKBACK, horizon=HORIZON)
    result, summary = evaluate(model, val_ds, panel, STATION, POLLUTANT, mu, sigma)


def run_all(lookback=168, horizon=24, epochs=20, device="cpu", out_csv="xlstm_results.csv"):
    """Loops over every 'ok' station-pollutant pair, trains one xLSTM each,
    and writes per-window scores to out_csv in the same shape as
    baseline_windows.csv — so this can be merged with baseline_metrics.csv's
    aggregation (bs.aggregate) for a single comparison table."""
    groups = bs.load_groups(SHARED_DIR)
    panel = bs.load_panel(CLEAN_CSV, SHARED_DIR)  # load once, reuse across pairs

    pairs = [(st, p) for p in bs.POLLUTANTS
              for st in groups["by_pollutant"][p]["ok"]]
    print(f"{len(pairs)} station-pollutant pairs to run")

    all_rows = []
    for station, pollutant in pairs:
        try:
            model, val_ds, _, mu, sigma = train_one_station(
                station, pollutant, lookback=lookback, horizon=horizon,
                epochs=epochs, device=device)
            result, _ = evaluate(model, val_ds, panel, station, pollutant, mu, sigma, device=device)
            all_rows.append(result)
        except Exception as e:
            print(f"SKIPPED {station}-{pollutant}: {e}")

    full = pd.concat(all_rows, ignore_index=True)
    full.to_csv(out_csv, index=False)

    per_pol = bs.aggregate(full.to_dict("records"))
    print("\nFULL SUMMARY — xLSTM, per pollutant, native units:\n")
    print(per_pol)
    print(f"\nwrote {out_csv} ({len(full)} rows)")
    print("Compare per_pol against baseline_metrics.csv row by row — remember "
          "SO2's bar is persistence_last, everything else is persistence_t24, "
          "and never average MAE/RMSE across pollutants (CO is mg/m3).")
    return full, per_pol


# to run the full sweep instead of the single-station smoke test above:
#   full, per_pol = run_all()

Downloading...
From (original): https://drive.google.com/uc?id=1QyfzghimgyRhQ735k42JetDEoK5kfGZJ
From (redirected): https://drive.google.com/uc?id=1QyfzghimgyRhQ735k42JetDEoK5kfGZJ&confirm=t&uuid=8d8565f7-1810-4deb-8d4d-99a0e03b99db
To: /content/bih_hourly_clean.csv
100%|██████████| 167M/167M [00:02<00:00, 62.1MB/s]
/content/shared/bih_shared.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "filled": g[f"{p}_filled"].fillna(False).to_numpy(bool)},
/content/shared/bih_shared.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "filled": g[f"{p}_filled

[Bjelave-pm10] epoch 1: train MAE (normalized)=14957.2622
[Bjelave-pm10] epoch 2: train MAE (normalized)=34.7970
[Bjelave-pm10] epoch 3: train MAE (normalized)=0.4311
[Bjelave-pm10] epoch 4: train MAE (normalized)=0.4199
[Bjelave-pm10] epoch 5: train MAE (normalized)=0.4145
[Bjelave-pm10] epoch 6: train MAE (normalized)=0.4125
[Bjelave-pm10] epoch 7: train MAE (normalized)=0.4129
[Bjelave-pm10] epoch 8: train MAE (normalized)=0.4087
[Bjelave-pm10] epoch 9: train MAE (normalized)=0.4081
[Bjelave-pm10] epoch 10: train MAE (normalized)=0.4060
[Bjelave-pm10] epoch 11: train MAE (normalized)=0.4053
[Bjelave-pm10] epoch 12: train MAE (normalized)=0.4057
[Bjelave-pm10] epoch 13: train MAE (normalized)=0.4035
[Bjelave-pm10] epoch 14: train MAE (normalized)=0.4031
[Bjelave-pm10] epoch 15: train MAE (normalized)=0.4021
[Bjelave-pm10] epoch 16: train MAE (normalized)=0.4000
[Bjelave-pm10] epoch 17: train MAE (normalized)=0.4036
[Bjelave-pm10] epoch 18: train MAE (normalized)=0.4053
[Bjelave-pm10]

In [ ]:
full, per_pol = run_all()


/content/shared/bih_shared.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "filled": g[f"{p}_filled"].fillna(False).to_numpy(bool)},
/content/shared/bih_shared.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "filled": g[f"{p}_filled"].fillna(False).to_numpy(bool)},
/content/shared/bih_shared.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcas

95 station-pollutant pairs to run


/content/shared/bih_shared.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "filled": g[f"{p}_filled"].fillna(False).to_numpy(bool)},
/content/shared/bih_shared.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "filled": g[f"{p}_filled"].fillna(False).to_numpy(bool)},
/content/shared/bih_shared.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcas

[Banja Luka-pm10] epoch 1: train MAE (normalized)=678.6786
[Banja Luka-pm10] epoch 2: train MAE (normalized)=0.4042
[Banja Luka-pm10] epoch 3: train MAE (normalized)=0.3827
[Banja Luka-pm10] epoch 4: train MAE (normalized)=0.3744
[Banja Luka-pm10] epoch 5: train MAE (normalized)=0.3702
[Banja Luka-pm10] epoch 6: train MAE (normalized)=0.3653
[Banja Luka-pm10] epoch 7: train MAE (normalized)=0.3620
[Banja Luka-pm10] epoch 8: train MAE (normalized)=0.3591
[Banja Luka-pm10] epoch 9: train MAE (normalized)=0.3589
[Banja Luka-pm10] epoch 10: train MAE (normalized)=0.3588
[Banja Luka-pm10] epoch 11: train MAE (normalized)=0.3564
[Banja Luka-pm10] epoch 12: train MAE (normalized)=0.3540
[Banja Luka-pm10] epoch 13: train MAE (normalized)=0.3535
[Banja Luka-pm10] epoch 14: train MAE (normalized)=0.3536
[Banja Luka-pm10] epoch 15: train MAE (normalized)=0.3527
[Banja Luka-pm10] epoch 16: train MAE (normalized)=0.3523
[Banja Luka-pm10] epoch 17: train MAE (normalized)=0.3502
[Banja Luka-pm10] epo

/content/shared/bih_shared.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "filled": g[f"{p}_filled"].fillna(False).to_numpy(bool)},
/content/shared/bih_shared.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "filled": g[f"{p}_filled"].fillna(False).to_numpy(bool)},
/content/shared/bih_shared.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcas

...

[Ilidza-so2] epoch 1: train MAE (normalized)=7016.9045
[Ilidza-so2] epoch 2: train MAE (normalized)=0.4618
[Ilidza-so2] epoch 3: train MAE (normalized)=1.7452
[Ilidza-so2] epoch 4: train MAE (normalized)=0.4226
[Ilidza-so2] epoch 5: train MAE (normalized)=0.4168
[Ilidza-so2] epoch 6: train MAE (normalized)=0.4132
[Ilidza-so2] epoch 7: train MAE (normalized)=0.4107
[Ilidza-so2] epoch 8: train MAE (normalized)=0.4086
[Ilidza-so2] epoch 9: train MAE (normalized)=1.0834
[Ilidza-so2] epoch 10: train MAE (normalized)=0.4059
[Ilidza-so2] epoch 11: train MAE (normalized)=0.4061
[Ilidza-so2] epoch 12: train MAE (normalized)=0.4041
[Ilidza-so2] epoch 13: train MAE (normalized)=0.4025
[Ilidza-so2] epoch 14: train MAE (normalized)=0.4044
[Ilidza-so2] epoch 15: train MAE (normalized)=0.4043
[Ilidza-so2] epoch 16: train MAE (normalized)=0.4036
[Ilidza-so2] epoch 17: train MAE (normalized)=0.4005
[Ilidza-so2] epoch 18: train MAE (normalized)=0.4022
[Ilidza-so2] epoch 19: train MAE (normalized)=0.4020

/content/shared/bih_shared.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "filled": g[f"{p}_filled"].fillna(False).to_numpy(bool)},
/content/shared/bih_shared.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "filled": g[f"{p}_filled"].fillna(False).to_numpy(bool)},
/content/shared/bih_shared.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcas

[Ilijas-so2] epoch 1: train MAE (normalized)=15684.7226
[Ilijas-so2] epoch 2: train MAE (normalized)=11.5614
[Ilijas-so2] epoch 3: train MAE (normalized)=0.3870
[Ilijas-so2] epoch 4: train MAE (normalized)=0.3754
[Ilijas-so2] epoch 5: train MAE (normalized)=0.3710
[Ilijas-so2] epoch 6: train MAE (normalized)=0.3690
[Ilijas-so2] epoch 7: train MAE (normalized)=0.3664
[Ilijas-so2] epoch 8: train MAE (normalized)=0.3639
[Ilijas-so2] epoch 9: train MAE (normalized)=0.3621
[Ilijas-so2] epoch 10: train MAE (normalized)=0.3610
[Ilijas-so2] epoch 11: train MAE (normalized)=0.3619
[Ilijas-so2] epoch 12: train MAE (normalized)=0.3582
[Ilijas-so2] epoch 13: train MAE (normalized)=0.3562
[Ilijas-so2] epoch 14: train MAE (normalized)=0.3584
[Ilijas-so2] epoch 15: train MAE (normalized)=0.3588
[Ilijas-so2] epoch 16: train MAE (normalized)=9.0028
[Ilijas-so2] epoch 17: train MAE (normalized)=0.3555
[Ilijas-so2] epoch 18: train MAE (normalized)=0.3560
[Ilijas-so2] epoch 19: train MAE (normalized)=0.35

/content/shared/bih_shared.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "filled": g[f"{p}_filled"].fillna(False).to_numpy(bool)},
/content/shared/bih_shared.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "filled": g[f"{p}_filled"].fillna(False).to_numpy(bool)},
/content/shared/bih_shared.py:62: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcas

[Isedlo-so2] epoch 1: train MAE (normalized)=21477.7804
[Isedlo-so2] epoch 2: train MAE (normalized)=0.4555
[Isedlo-so2] epoch 3: train MAE (normalized)=0.4032
[Isedlo-so2] epoch 4: train MAE (normalized)=0.3854
[Isedlo-so2] epoch 5: train MAE (normalized)=0.3789
[Isedlo-so2] epoch 6: train MAE (normalized)=0.3762
[Isedlo-so2] epoch 7: train MAE (normalized)=0.3736
[Isedlo-so2] epoch 8: train MAE (normalized)=0.3727
[Isedlo-so2] epoch 9: train MAE (normalized)=0.3707
[Isedlo-so2] epoch 10: train MAE (normalized)=0.3705
[Isedlo-so2] epoch 11: train MAE (normalized)=0.3682
[Isedlo-so2] epoch 12: train MAE (normalized)=0.3709
[Isedlo-so2] epoch 13: train MAE (normalized)=0.3698
[Isedlo-so2] epoch 14: train MAE (normalized)=0.3669
[Isedlo-so2] epoch 15: train MAE (normalized)=0.3664
[Isedlo-so2] epoch 16: train MAE (normalized)=0.3650
[Isedlo-so2] epoch 17: train MAE (normalized)=0.3661
[Isedlo-so2] epoch 18: train MAE (normalized)=0.3662
[Isedlo-so2] epoch 19: train MAE (normalized)=0.364